# BirdCLEF 2026 — 4-Model Ensemble (Perch TFLite × 3 + SED PyTorch)

**Model 1**: Perch labels → gather(234) → `label_head_pseudo.tflite`  
**Model 2**: Perch labels → gather(234) → `label_head_soundscape_train.tflite`  
**Model 3**: Perch embedding (1536) → `embedding_head.tflite`  
**Model 4**: Raw audio → Mel → EfficientNet-B0 SED (PyTorch)  
**Ensemble**: Weighted average (Perch × 3 weight=1.0 each, SED weight=1.0)  
**Human voice removal**: Silero VAD  
**Post-processing**: threshold(0.02)  
**Speed**: ~0.25s/clip → ~9 min for 600 soundscapes (4 threads)  

Required datasets:
- `birdclef2026-ensemble-weights` — all TFLite + PyTorch weights + Silero VAD
- `models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1` — Perch labels.csv

In [ ]:
import subprocess, sys

# TF required for TFLite interpreter
try:
    import tensorflow as tf
    assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 17)
except (ImportError, AssertionError):
    import os
    tb  = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl'
    tf_ = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
    for w in [tb, tf_]:
        if os.path.isfile(w):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', w], check=True)
    import tensorflow as tf
    print(f'TF upgraded to {tf.__version__}')

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow')

import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio.transforms as T
import timm
import threading, glob, os, re, time
import librosa, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

START = time.time()
print(f'TF {tf.__version__}  PyTorch {torch.__version__}')

## Config

In [ ]:
DATA_DIR     = '/kaggle/input/birdclef-2026'
PERCH_DIR    = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'
DATASET_DIR  = '/kaggle/input/birdclef2026-ensemble-weights'

# ── TFLite paths ──────────────────────────────────────────────────────────────
PERCH_TFLITE    = f'{DATASET_DIR}/perch_v2_cpu.tflite'
HEAD_PSEUDO     = f'{DATASET_DIR}/label_head_pseudo.tflite'
HEAD_SOUNDSCAPE = f'{DATASET_DIR}/label_head_soundscape_train.tflite'
HEAD_EMBEDDING  = f'{DATASET_DIR}/embedding_head_nohuman_embedding_soundscape.tflite'

# ── SED PyTorch path ──────────────────────────────────────────────────────────
SED_CHECKPOINT  = f'{DATASET_DIR}/best_sed_b0_v5.pt'
SILERO_DIR      = f'{DATASET_DIR}/silero_vad'

# ── Ensemble weights (tune based on holdout eval) ─────────────────────────────
W_PSEUDO     = 1.0   # label-head pseudo
W_SOUNDSCAPE = 1.0   # label-head soundscape-train
W_EMBEDDING  = 1.0   # embedding-head
W_SED        = 1.0   # SED
W_TOTAL      = W_PSEUDO + W_SOUNDSCAPE + W_EMBEDDING + W_SED

# Use SED only if checkpoint exists
USE_SED = os.path.isfile(SED_CHECKPOINT)
USE_EMB = os.path.isfile(HEAD_EMBEDDING)
print(f'USE_SED={USE_SED}  USE_EMB={USE_EMB}')

NUM_CLASSES  = 234
N_PERCH      = 14795
N_EMBED      = 1536
SR           = 32_000
CLIP_SAMPLES = SR * 5
PP_THRESHOLD = 0.02

TEST_DIR  = os.path.join(DATA_DIR, 'test_soundscapes')
SC_DIR    = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else os.path.join(DATA_DIR, 'train_soundscapes')
ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
primary_labels = sample_sub.columns[1:].tolist()
print(f'Soundscapes: {len(ogg_files)}  |  Species: {len(primary_labels)}')

## SED Model Definition

In [ ]:
@dataclass
class SEDConfig:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0


class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob  = torch.sigmoid(clipwise_logit)
        return {'clipwise_prob': clipwise_prob, 'segmentwise_logit': cls.permute(0, 2, 1)}


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=False,
            in_chans=cfg.in_channels, features_only=False,
            global_pool='', num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled   = self.gem_pool(features)
        return self.head(pooled)


class MelTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        # Peak-normalize per sample (matches apply_gpu_mel in training)
        peak = waveforms.abs().amax(dim=1, keepdim=True).clamp(min=1e-7)
        waveforms = waveforms / peak
        mel = self.db(self.mel(waveforms))
        B   = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min  = mel_flat.min(1, keepdim=True)[0].unsqueeze(-1)
        mel_max  = mel_flat.max(1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)  # (B, 3, n_mels, T)

print('SED model classes defined.')

## Species Mapping (Perch → BirdCLEF 234)

In [ ]:
bc_labels = pd.read_csv(os.path.join(PERCH_DIR, 'assets/labels.csv'))
bc_labels = (bc_labels.reset_index()
             .rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1)
             .set_index('scientific_name'))

taxonomy  = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))
mapping   = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(N_PERCH).astype(int)
label_indices_np = mapping.set_index('primary_label')['bc_index'].reindex(primary_labels).fillna(N_PERCH).astype(int).to_numpy()
print(f'Perch classes: {N_PERCH}  mapped: {(label_indices_np < N_PERCH).sum()}/234')

## Load TFLite Interpreters (thread-local)

In [ ]:
_tls = threading.local()

def get_tflite_interps():
    """Thread-local TFLite interpreters (Perch + 3 heads)."""
    if not hasattr(_tls, 'perch'):
        p = tf.lite.Interpreter(model_path=PERCH_TFLITE, num_threads=1)
        p.allocate_tensors()
        p_out = p.get_output_details()
        p_lbl = next(i for i, o in enumerate(p_out) if o['shape'][-1] == N_PERCH)
        p_emb = next(i for i, o in enumerate(p_out) if o['shape'][-1] == N_EMBED)
        _tls.perch      = p
        _tls.p_inp_idx  = p.get_input_details()[0]['index']
        _tls.p_lbl_idx  = p_out[p_lbl]['index']
        _tls.p_emb_idx  = p_out[p_emb]['index']

        for attr, path in [('h1', HEAD_PSEUDO), ('h2', HEAD_SOUNDSCAPE)]:
            h = tf.lite.Interpreter(model_path=path, num_threads=1)
            h.allocate_tensors()
            setattr(_tls, attr,          h)
            setattr(_tls, attr + '_inp', h.get_input_details()[0]['index'])
            setattr(_tls, attr + '_out', h.get_output_details()[0]['index'])

        if USE_EMB:
            h3 = tf.lite.Interpreter(model_path=HEAD_EMBEDDING, num_threads=1)
            h3.allocate_tensors()
            _tls.h3     = h3
            _tls.h3_inp = h3.get_input_details()[0]['index']
            _tls.h3_out = h3.get_output_details()[0]['index']
    return _tls


# Validate on main thread
tls = get_tflite_interps()
dummy = np.zeros((1, CLIP_SAMPLES), dtype=np.float32)
tls.perch.set_tensor(tls.p_inp_idx, dummy); tls.perch.invoke()
lbl = tls.perch.get_tensor(tls.p_lbl_idx)[0]
emb = tls.perch.get_tensor(tls.p_emb_idx)[0]
padded  = np.append(lbl, 0.0).astype(np.float32)[label_indices_np][None]
tls.h1.set_tensor(tls.h1_inp, padded); tls.h1.invoke(); out1 = tls.h1.get_tensor(tls.h1_out)
tls.h2.set_tensor(tls.h2_inp, padded); tls.h2.invoke(); out2 = tls.h2.get_tensor(tls.h2_out)
print(f'Perch→labels{lbl.shape} embed{emb.shape}  head1{out1.shape}  head2{out2.shape}', end='')
if USE_EMB:
    tls.h3.set_tensor(tls.h3_inp, emb[None]); tls.h3.invoke(); out3 = tls.h3.get_tensor(tls.h3_out)
    print(f'  head3{out3.shape}', end='')
print(f'  OK')

## Load SED Model (PyTorch, CPU)

In [ ]:
_sed_model = None
_sed_mel   = None
_sed_cfg   = SEDConfig()
_SED_LOCK  = threading.Lock()

if USE_SED:
    print(f'Loading SED: {SED_CHECKPOINT}')
    _sed_model = SEDModel(_sed_cfg)
    ckpt = torch.load(SED_CHECKPOINT, map_location='cpu', weights_only=False)
    _sed_model.load_state_dict(ckpt['model_state_dict'])
    _sed_model.eval()
    _sed_mel = MelTransform(_sed_cfg)
    _sed_mel.eval()
    print(f'  Loaded epoch {ckpt["epoch"]}  val_auc={ckpt["metrics"].get("macro_auc", "?")}')
    # Warm-up
    with torch.no_grad():
        _d = torch.zeros(1, CLIP_SAMPLES)
        _m = _sed_mel(_d)
        _o = _sed_model(_m)
    print(f'  SED warm-up OK → {_o["clipwise_prob"].shape}')
else:
    print(f'SED checkpoint not found: {SED_CHECKPOINT} — SED disabled')


def sed_predict_clip(clip_np):
    """Run SED on a (160000,) float32 clip. Returns (234,) probs."""
    if _sed_model is None:
        return None
    t = torch.from_numpy(clip_np).float().unsqueeze(0)   # (1, 160000)
    with _SED_LOCK, torch.no_grad():
        mel  = _sed_mel(t)
        out  = _sed_model(mel)
    clip_prob = out[0] if isinstance(out, tuple) else out
    return clip_prob[0].numpy()                            # (234,)

## Human Voice Removal (Silero VAD)

In [ ]:
_vad_model, _vad_utils = torch.hub.load(
    repo_or_dir=SILERO_DIR, model='silero_vad', source='local', verbose=False)
get_speech_timestamps = _vad_utils[0]
_VAD_LOCK = threading.Lock()
print('Silero VAD loaded')

def remove_human_voice(audio, sr=32_000):
    SPEECH_MIN_DUR  = 2.0
    SPEECH_START_TH = 8.0
    THRESHOLD       = 0.4
    t16 = torch.tensor(librosa.resample(audio, orig_sr=sr, target_sr=16_000))
    with _VAD_LOCK:
        segs = get_speech_timestamps(t16, _vad_model, threshold=THRESHOLD, sampling_rate=16_000)
    for seg in segs:
        start_s = seg['start'] / 16_000
        dur_s   = (seg['end'] - seg['start']) / 16_000
        if dur_s >= SPEECH_MIN_DUR and start_s >= SPEECH_START_TH:
            return audio[:int(start_s * sr)]
    return audio

## Inference (4-model ensemble per clip)

In [ ]:
def predict_file(ogg_path):
    ss_id   = re.sub(r'\.ogg$', '', os.path.basename(ogg_path), flags=re.IGNORECASE)
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]
    zeros   = np.zeros((12, NUM_CLASSES), dtype=np.float32)
    try:
        audio, _ = librosa.load(ogg_path, sr=SR, mono=True)
        audio = audio.astype(np.float32)
        audio = remove_human_voice(audio, sr=SR)
    except Exception as e:
        print(f'  ERROR {os.path.basename(ogg_path)}: {e}')
        return row_ids, zeros

    tls   = get_tflite_interps()
    preds = np.zeros((12, NUM_CLASSES), dtype=np.float32)

    for i in range(12):
        start = i * CLIP_SAMPLES
        clip  = audio[start: start + CLIP_SAMPLES]
        if len(clip) < CLIP_SAMPLES:
            clip = np.pad(clip, (0, CLIP_SAMPLES - len(clip)))

        # ── Perch TFLite (single forward → both label + embedding) ────────────
        tls.perch.set_tensor(tls.p_inp_idx, clip[None])
        tls.perch.invoke()
        label_logits = tls.perch.get_tensor(tls.p_lbl_idx)[0]          # (14795,)
        embedding    = tls.perch.get_tensor(tls.p_emb_idx)[0]          # (1536,)

        padded   = np.append(label_logits, 0.0).astype(np.float32)
        gathered = padded[label_indices_np][None]                       # (1, 234)

        # ── Model 1: label-head pseudo ────────────────────────────────────────
        tls.h1.set_tensor(tls.h1_inp, gathered); tls.h1.invoke()
        pred1 = tls.h1.get_tensor(tls.h1_out)[0]                       # (234,)

        # ── Model 2: label-head soundscape-train ──────────────────────────────
        tls.h2.set_tensor(tls.h2_inp, gathered); tls.h2.invoke()
        pred2 = tls.h2.get_tensor(tls.h2_out)[0]                       # (234,)

        # ── Model 3: embedding-head ────────────────────────────────────────────
        pred3 = np.zeros(NUM_CLASSES, dtype=np.float32)
        if USE_EMB:
            tls.h3.set_tensor(tls.h3_inp, embedding[None]); tls.h3.invoke()
            pred3 = tls.h3.get_tensor(tls.h3_out)[0]                   # (234,)

        # ── Model 4: SED ───────────────────────────────────────────────────────
        pred4 = np.zeros(NUM_CLASSES, dtype=np.float32)
        if USE_SED:
            pred4 = sed_predict_clip(clip)

        # ── Weighted average ──────────────────────────────────────────────────
        total_w = W_PSEUDO + W_SOUNDSCAPE + (W_EMBEDDING if USE_EMB else 0) + (W_SED if USE_SED else 0)
        preds[i] = (W_PSEUDO  * pred1 + W_SOUNDSCAPE * pred2 +
                   (W_EMBEDDING * pred3 if USE_EMB else 0) +
                   (W_SED       * pred4 if USE_SED else 0)) / total_w

    return row_ids, preds


NUM_THREADS = 4 if not USE_SED else 2   # reduce threads if SED is CPU-heavy
print(f'Inferring {len(ogg_files)} soundscapes  models: label×2 + {"emb" if USE_EMB else ""} + {"SED" if USE_SED else ""}  threads={NUM_THREADS}')
all_row_ids, all_preds = [], []
with ThreadPoolExecutor(max_workers=NUM_THREADS) as ex:
    for k, (rids, pred) in enumerate(ex.map(predict_file, ogg_files)):
        all_row_ids.extend(rids)
        all_preds.append(pred)
        if k % 50 == 0:
            elapsed = (time.time() - START) / 60
            eta = elapsed / (k + 1) * (len(ogg_files) - k - 1) if k > 0 else 0
            print(f'  [{k+1}/{len(ogg_files)}] {elapsed:.1f} min  eta={eta:.1f} min')

predictions = np.concatenate(all_preds, axis=0)
print(f'Done: {len(all_row_ids)} rows  shape={predictions.shape}  total={(time.time()-START)/60:.1f} min')

## Post-Processing + Save Submission

In [ ]:
predictions_pp = predictions.copy()
predictions_pp[predictions_pp < PP_THRESHOLD] = 0.0
print(f'Post-processing: threshold={PP_THRESHOLD}  zeros={(predictions_pp==0).mean():.1%}')

submission = pd.DataFrame(predictions_pp, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape[0]} rows x {len(primary_labels)} species')
print(f'Total elapsed: {(time.time()-START)/60:.1f} min')
submission.head(3)